# MagFace Cosine Verification
Pipeline:
1. Crop faces using RetinaFace
2. Extract embeddings using MagFace (via InsightFace)
3. Verify using cosine similarity threshold = 0.6
4. Report Accuracy, Precision, Recall, F1, FNMR, FMR

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# install libraries
%pip install retina-face insightface onnxruntime-gpu tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 103.9 MB/s eta 0:00:00


In [5]:
# unzip celebrity dataset from drive
import zipfile
print("Unzipping celebrity_dataset...")
with zipfile.ZipFile('/content/drive/MyDrive/celebrity_dataset.zip', 'r') as z:
    z.extractall('/content/')
print("Done!")

Unzipping celebrity_dataset...
Done!


In [6]:
# check structure
import os
print("Gallery identities:", len(os.listdir('/content/celebrity_dataset/gallery')))
print("Probe identities:",   len(os.listdir('/content/celebrity_dataset/probe')))
print("Sample:", os.listdir('/content/celebrity_dataset/gallery')[:3])

Gallery identities: 100
Probe identities: 100
Sample: ['Hugo_Weaving', 'Hal_Holbrook', 'Sally_Field']


In [7]:
# imports
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

from retinaface import RetinaFace
from insightface.app import FaceAnalysis
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [8]:
# paths
GALLERY_DIR        = '/content/celebrity_dataset/gallery'
PROBE_DIR          = '/content/celebrity_dataset/probe'
GALLERY_EMBED_DIR  = '/content/celebrity_dataset/gallery_embeddings'
PROBE_EMBED_DIR    = '/content/celebrity_dataset/probe_embeddings'
GALLERY_CROP_DIR   = '/content/celebrity_dataset/gallery_cropped'

for folder in [GALLERY_EMBED_DIR, PROBE_EMBED_DIR, GALLERY_CROP_DIR]:
    os.makedirs(folder, exist_ok=True)

THRESHOLD = 0.6
print("Paths ready! Threshold =", THRESHOLD)

Paths ready! Threshold = 0.6


In [10]:
# load MagFace via InsightFace
print("Loading model...")
app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0, det_size=(640, 640))
rec_model = app.models['recognition']
print("Model loaded!")

Loading model...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 

In [11]:
# helper function: crop face using RetinaFace
def crop_face(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    try:
        faces = RetinaFace.detect_faces(img_path)
        if isinstance(faces, dict) and len(faces) > 0:
            best = max(faces.values(), key=lambda f: f['score'])
            x1, y1, x2, y2 = best['facial_area']
            h, w = img.shape[:2]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            face = img[y1:y2, x1:x2]
            if face.size > 0:
                return cv2.resize(face, (112, 112))
    except:
        pass
    # fallback: resize whole image
    return cv2.resize(img, (112, 112))

# helper function: get embedding from cropped face
def get_embedding(face_img):
    face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    embedding = rec_model.get_feat(face_rgb).flatten().astype(np.float32)
    # L2 normalise
    norm = np.linalg.norm(embedding)
    if norm > 0:
        embedding = embedding / norm
    return embedding

print("Helper functions ready!")

Helper functions ready!


In [12]:
# STEP 1: extract gallery embeddings
print("Extracting gallery embeddings...")
gallery_saved = 0
gallery_failed = 0

for identity in tqdm(os.listdir(GALLERY_DIR)):
    identity_dir = os.path.join(GALLERY_DIR, identity)
    if not os.path.isdir(identity_dir):
        continue

    save_dir = os.path.join(GALLERY_EMBED_DIR, identity)
    os.makedirs(save_dir, exist_ok=True)

    for img_file in os.listdir(identity_dir):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            continue

        img_path = os.path.join(identity_dir, img_file)
        face = crop_face(img_path)

        if face is None:
            gallery_failed += 1
            continue

        embedding = get_embedding(face)
        base_name = os.path.splitext(img_file)[0]
        np.save(os.path.join(save_dir, base_name + '.npy'), embedding)
        gallery_saved += 1

print(f"Gallery embeddings saved: {gallery_saved}")
print(f"Gallery failed: {gallery_failed}")

Extracting gallery embeddings...


  0%|          | 0/100 [00:00<?, ?it/s]

26-07-06 05:03:34 - Directory /root/.deepface created
26-07-06 05:03:34 - Directory /root/.deepface/weights created
26-07-06 05:03:34 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5

  0%|          | 0.00/119M [00:00<?, ?B/s]
  9%|▉         | 11.0M/119M [00:00<00:01, 78.9MB/s]
 18%|█▊        | 21.5M/119M [00:00<00:01, 91.4MB/s]
 27%|██▋       | 32.5M/119M [00:00<00:00, 98.2MB/s]
 36%|███▌      | 43.0M/119M [00:00<00:00, 91.3MB/s]
 45%|████▍     | 53.0M/119M [00:00<00:00, 93.0MB/s]
 53%|█████▎    | 63.4M/119M [00:00<00:00, 96.3MB/s]
 62%|██████▏   | 73.9M/119M [00:00<00:00, 92.8MB/s]
 73%|███████▎  | 86.5M/119M [00:00<00:00, 102MB/s] 
 82%|████████▏ | 97.0M/119M [00:01<00:00, 94.5MB/s]
 90%|█████████ | 107M/119M [00:01<00:00, 87.7MB/s] 
100%|██████████| 119M/119M [00:01<00:00, 91.7MB/s]
100%|██████████| 100/100 [01:01<00:00,  1.63it/s]

Gallery embeddings saved: 100
Gallery failed: 0


In [13]:
# STEP 2: extract probe embeddings
print("Extracting probe embeddings...")
probe_saved = 0
probe_failed = 0

for identity in tqdm(os.listdir(PROBE_DIR)):
    identity_dir = os.path.join(PROBE_DIR, identity)
    if not os.path.isdir(identity_dir):
        continue

    save_dir = os.path.join(PROBE_EMBED_DIR, identity)
    os.makedirs(save_dir, exist_ok=True)

    for img_file in os.listdir(identity_dir):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            continue

        img_path = os.path.join(identity_dir, img_file)
        face = crop_face(img_path)

        if face is None:
            probe_failed += 1
            continue

        embedding = get_embedding(face)
        base_name = os.path.splitext(img_file)[0]
        np.save(os.path.join(save_dir, base_name + '.npy'), embedding)
        probe_saved += 1

print(f"Probe embeddings saved: {probe_saved}")
print(f"Probe failed: {probe_failed}")

Extracting probe embeddings...


100%|██████████| 100/100 [00:51<00:00,  1.95it/s]

Probe embeddings saved: 100
Probe failed: 0


In [14]:
# STEP 3: load gallery database
def load_gallery_db(gallery_embed_root):
    gallery_db = {}
    total = 0
    for identity in os.listdir(gallery_embed_root):
        identity_dir = os.path.join(gallery_embed_root, identity)
        if not os.path.isdir(identity_dir):
            continue
        gallery_db[identity] = []
        for emb_file in os.listdir(identity_dir):
            if not emb_file.endswith('.npy'):
                continue
            emb = np.load(os.path.join(identity_dir, emb_file))
            gallery_db[identity].append(emb)
            total += 1
    print(f"Gallery loaded: {len(gallery_db)} identities, {total} embeddings")
    return gallery_db

gallery_db = load_gallery_db(GALLERY_EMBED_DIR)

Gallery loaded: 100 identities, 100 embeddings


In [15]:
# STEP 4: cosine verification
results = []

for identity in tqdm(os.listdir(PROBE_EMBED_DIR)):
    probe_identity_dir = os.path.join(PROBE_EMBED_DIR, identity)
    if not os.path.isdir(probe_identity_dir):
        continue

    for emb_file in os.listdir(probe_identity_dir):
        if not emb_file.endswith('.npy'):
            continue

        probe_emb = np.load(os.path.join(probe_identity_dir, emb_file))

        # compare against all gallery identities
        identity_scores = {}
        for gal_identity, gal_embeddings in gallery_db.items():
            sims = [
                cosine_similarity(
                    probe_emb.reshape(1, -1),
                    g.reshape(1, -1)
                )[0][0]
                for g in gal_embeddings
            ]
            identity_scores[gal_identity] = max(sims)

        # get best match
        sorted_scores = sorted(
            identity_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
        best_identity  = sorted_scores[0][0]
        best_similarity = sorted_scores[0][1]

        # cosine threshold decision
        decision = 1 if best_similarity >= THRESHOLD else 0

        # ground truth label
        # 1 if probe identity is in gallery and best match is correct
        label = 1 if (identity in gallery_db and best_identity == identity) else 0

        results.append({
            'true_identity':    identity,
            'predicted':        best_identity,
            'best_similarity':  best_similarity,
            'decision':         decision,
            'label':            label,
        })

print(f"Total probes tested: {len(results)}")

100%|██████████| 100/100 [00:03<00:00, 28.15it/s]

Total probes tested: 100


In [16]:
# STEP 5: compute metrics
df = pd.DataFrame(results)

y_true = df['label']
y_pred = df['decision']

# confusion matrix values
cm = confusion_matrix(y_true, y_pred)
TN, FP, FN, TP = cm.ravel()

# metrics
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)

# biometric metrics
FNMR = FN / (TP + FN) if (TP + FN) > 0 else 0  # False Non-Match Rate
FMR  = FP / (TN + FP) if (TN + FP) > 0 else 0  # False Match Rate
TAR  = TP / (TP + FN) if (TP + FN) > 0 else 0  # True Acceptance Rate
TRR  = TN / (TN + FP) if (TN + FP) > 0 else 0  # True Rejection Rate

print("====== RESULTS (MagFace + Cosine Threshold=0.6) ======")
print(f"Accuracy:              {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"Precision:             {precision:.4f}")
print(f"Recall (genuine acc):  {recall:.4f}")
print(f"F1-score:              {f1:.4f}")
print(f"FNMR (genuine degrad): {FNMR:.4f}")
print(f"FMR  (impostor):       {FMR:.4f}")
print()
print("====== CONFUSION MATRIX ======")
print(f"TP: {TP}  TN: {TN}  FP: {FP}  FN: {FN}")
print()
print("====== ADDITIONAL METRICS ======")
print(f"TAR (True Accept Rate):  {TAR:.4f}")
print(f"TRR (True Reject Rate):  {TRR:.4f}")

====== RESULTS (MagFace + Cosine Threshold=0.6) ======
Accuracy:              0.1600  (16.00%)
Precision:             1.0000
Recall (genuine acc):  0.0345
F1-score:              0.0667
FNMR (genuine degrad): 0.9655
FMR  (impostor):       0.0000

====== CONFUSION MATRIX ======
TP: 3  TN: 13  FP: 0  FN: 84

====== ADDITIONAL METRICS ======
TAR (True Accept Rate):  0.0345
TRR (True Reject Rate):  1.0000


In [25]:
# STEP 6: save results to drive
DRIVE_SAVE = '/content/drive/MyDrive'

df.to_csv(f'{DRIVE_SAVE}/magface_cosine_results.csv', index=False)
print("Results saved to Drive!")

# save metrics summary
with open(f'{DRIVE_SAVE}/magface_cosine_metrics.txt', 'w') as f:
    f.write("MagFace + Cosine Threshold=0.6\n")
    f.write("==============================\n")
    f.write(f"Accuracy:              {accuracy:.4f}\n")
    f.write(f"Precision:             {precision:.4f}\n")
    f.write(f"Recall (genuine acc):  {recall:.4f}\n")
    f.write(f"F1-score:              {f1:.4f}\n")
    f.write(f"FNMR (genuine degrad): {FNMR:.4f}\n")
    f.write(f"FMR  (impostor):       {FMR:.4f}\n")
    f.write(f"TP: {TP}  TN: {TN}  FP: {FP}  FN: {FN}\n")
print("Metrics saved to Drive!")

Results saved to Drive!
Metrics saved to Drive!
